In [27]:
data_abs_path = "/home/sxi219/RAI-Account/MSR_data_cleaned.csv"


In [28]:
import pandas as pd

df = pd.read_csv(data_abs_path, nrows=1000)

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 36 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Unnamed: 0                    1000 non-null   int64  
 1   Access Gained                 252 non-null    object 
 2   Attack Origin                 1000 non-null   object 
 3   Authentication Required       1000 non-null   object 
 4   Availability                  965 non-null    object 
 5   CVE ID                        1000 non-null   object 
 6   CVE Page                      1000 non-null   object 
 7   CWE ID                        913 non-null    object 
 8   Complexity                    1000 non-null   object 
 9   Confidentiality               621 non-null    object 
 10  Integrity                     557 non-null    object 
 11  Known Exploits                0 non-null      float64
 12  Publish Date                  1000 non-null   object 
 13  Scor

In [30]:
print(df.columns)

Index(['Unnamed: 0', 'Access Gained', 'Attack Origin',
       'Authentication Required', 'Availability', 'CVE ID', 'CVE Page',
       'CWE ID', 'Complexity', 'Confidentiality', 'Integrity',
       'Known Exploits', 'Publish Date', 'Score', 'Summary', 'Update Date',
       'Vulnerability Classification', 'add_lines', 'codeLink', 'commit_id',
       'commit_message', 'del_lines', 'file_name', 'files_changed',
       'func_after', 'func_before', 'lang', 'lines_after', 'lines_before',
       'parentID', 'patch', 'project', 'project_after', 'project_before',
       'vul', 'vul_func_with_fix'],
      dtype='object')


In [31]:
df_code_after = df['func_after']
df_code_lang = df['lang'].str.lower()

In [32]:
df_code_after.info()

<class 'pandas.core.series.Series'>
RangeIndex: 1000 entries, 0 to 999
Series name: func_after
Non-Null Count  Dtype 
--------------  ----- 
1000 non-null   object
dtypes: object(1)
memory usage: 7.9+ KB


# IPAG Builder Experiment

In [33]:
from src.graph.ipag_builder import IPAGBuilder
from src.graph.build_language import LanguageBuilder

langs = set(df_code_lang.str.lower())
lang_map = LanguageBuilder(langs)
ipag = IPAGBuilder(source = df_code_after, language = df_code_lang.str.lower(), lang_map=lang_map.build())
ipag.build()
df_ipag = ipag.get_ipag_dataframe()


Building ASTs for all code snippets


Processed 100/1000 snippets...
Processed 200/1000 snippets...
Processed 300/1000 snippets...
Processed 400/1000 snippets...
Processed 500/1000 snippets...
Processed 600/1000 snippets...
Processed 700/1000 snippets...
Processed 800/1000 snippets...
Processed 900/1000 snippets...
Processed 1000/1000 snippets...
Finished building ASTs
Successful: 1000, Failed: 0, Total: 1000
IPAG Construction Complete
Total snippets: 1000
Average nodes per snippet: 261.3
Average edges per snippet: 260.3


## Building Node features for GIN training

In [34]:
# from src.graph.features import BuildNodeFeatures

# builder = BuildNodeFeatures(device='cuda')

all_ipag_nodes = df_ipag['ipag_nodes'].tolist()
all_ipag_edges = df_ipag['ipag_edges'].tolist()

# features_batch = builder.process_ipag_batch(all_ipag_nodes, all_ipag_edges)


In [35]:
# builder.save_features(features=features_batch, filepath="/home/sxi219/RAI-Account/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer/data/processed/1000.pkl")

In [36]:
# Test script: test_acceleration.py
import sys
sys.path.insert(0, 'src')

from src.graph import BuildNodeFeaturesAccelerated, ACCELERATED_AVAILABLE

print(f"Accelerated module available: {ACCELERATED_AVAILABLE}")

if ACCELERATED_AVAILABLE:
    builder = BuildNodeFeaturesAccelerated(batch_size=128)
    print("✓ BuildNodeFeaturesAccelerated initialized successfully")
    
    # Test with dummy data
    nodes = [
        {'id': 'n1', 'type': 'TOKEN', 'label': 'int'},
        {'id': 'n2', 'type': 'DECLARATION', 'label': 'x'},
        {'id': 'n3', 'type': 'PROPERTY', 'label': 'value'},
    ]
    edges = [
        {'source': 'n1', 'target': 'n2'},
        {'source': 'n2', 'target': 'n3'},
    ]
    
    features = builder.get_node_features(nodes, edges)
    print(f"✓ Extracted features for {len(features)} nodes")
    print(f"✓ Feature dimension: {features['n1']['combined'].shape}")
else:
    print("✗ Accelerated module not available")

Accelerated module available: False
✗ Accelerated module not available


In [37]:
from src.graph.features_accelerated import BuildNodeFeaturesAccelerated

builder = BuildNodeFeaturesAccelerated(device='cuda', batch_size=128)

all_ipag_nodes = df_ipag['ipag_nodes'].tolist()
all_ipag_edges = df_ipag['ipag_edges'].tolist()

features_batch = builder.process_ipag_batch(all_ipag_nodes, all_ipag_edges)

ModuleNotFoundError: No module named 'graph'

In [ ]:
from src.graph import graph_structure_features

